In [1]:
!pip install -U mljar-supervised

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.5/98.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 85.7 MB/s eta 0:00:00
  Created wheel for mljar-supervised: filename=mljar_supervised-1.1.17-py3-none-any.whl size=164663 sha256=273d17e95538c1bf238977775d90a3cd2dc247fc7787b42b3d000f6225fad468
  Stored in directory: /root/.cache/pip/wheels/6c/65/96/b05fb5e270b0c3cf376990dc459932cd056e6c179467699a81
  Created wheel for mljar-scikit-plot: filename=mljar_scikit_plot-0.3.12-py3-none-any.whl size=32014 sha256=11f3151cfd3944ca50a39d25729c310d86a4ed70b2620ae6a9cf37f7db486453
  Stored in directory: /root/.cache/pip/wheels/e2/2d/3e/8afe0632e7b03c3ae7b8048f88f0dcca4355b32d31abaea3ba
Successfully built mljar-supervised mljar-

In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [3]:
from supervised.automl import AutoML

In [4]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [5]:
train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv').drop(columns=['id'])
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [6]:
train['humidity'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['humidity'].to_numpy()])
train['pressure'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['pressure'].to_numpy()])
train['wind_speed'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['wind_speed'].to_numpy()])

test['humidity'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['humidity'].to_numpy()])
test['pressure'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['pressure'].to_numpy()])
test['wind_speed'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['wind_speed'].to_numpy()])

column = list(train.columns)
column.remove('efficiency')
column.remove('string_id')
column.remove('error_code')
column.remove('installation_type')

for i_ in range(len(column)):
    for j in column[i_:]:
        i = column[i_]
        if i != j:
            print(f'{i} x {j}')
            train[f'{i} x {j}'] = (train[i].astype('float').to_numpy() * train[j].astype('float').to_numpy())
            test[f'{i} x {j}'] = (test[i].astype('float').to_numpy() * test[j].astype('float').to_numpy())

temperature x irradiance
temperature x humidity
temperature x panel_age
temperature x maintenance_count
temperature x soiling_ratio
temperature x voltage
temperature x current
temperature x module_temperature
temperature x cloud_coverage
temperature x wind_speed
temperature x pressure
irradiance x humidity
irradiance x panel_age
irradiance x maintenance_count
irradiance x soiling_ratio
irradiance x voltage
irradiance x current
irradiance x module_temperature
irradiance x cloud_coverage
irradiance x wind_speed
irradiance x pressure
humidity x panel_age
humidity x maintenance_count
humidity x soiling_ratio
humidity x voltage
humidity x current
humidity x module_temperature
humidity x cloud_coverage
humidity x wind_speed
humidity x pressure
panel_age x maintenance_count
panel_age x soiling_ratio
panel_age x voltage
panel_age x current
panel_age x module_temperature
panel_age x cloud_coverage
panel_age x wind_speed
panel_age x pressure
maintenance_count x soiling_ratio
maintenance_count x 

In [7]:
automl = AutoML(
    mode="Compete", 
    eval_metric="rmse",
    model_time_limit=5*60,
    total_time_limit=60*60,
    top_models_to_improve=3,
    start_random_models=5,
    hill_climbing_steps=0,
    features_selection=True,
    golden_features=False,
    stack_models=True,
    train_ensemble=True,
    validation_strategy={
        "validation_type": "kfold",
        "k_folds": 3,
        "shuffle": True,
        "stratify": True,
    },
    algorithms=['Random Forest', 
                'Extra Trees', 
                'Xgboost', 
                'CatBoost', 
                'Neural Network', 
                'Nearest Neighbors']
)

automl.fit(train.drop(columns=['efficiency']), train["efficiency"])

AutoML directory: AutoML_1
The task is regression with evaluation metric rmse
AutoML will use algorithms: ['Random Forest', 'Extra Trees', 'Xgboost', 'CatBoost', 'Neural Network', 'Nearest Neighbors']
AutoML will stack models
AutoML will ensemble available models
AutoML steps: ['simple_algorithms', 'default_algorithms', 'not_so_random', 'mix_encoding', 'kmeans_features', 'insert_random_feature', 'features_selection', 'boost_on_errors', 'ensemble', 'stack', 'ensemble_stacked']
Skip simple_algorithms because no parameters were generated.
* Step default_algorithms will try to check up to 6 models
1_Default_Xgboost rmse 0.104141 trained in 8.35 seconds
2_Default_CatBoost rmse 0.103516 trained in 12.55 seconds
3_Default_NeuralNetwork rmse 0.104611 trained in 5.44 seconds
4_Default_RandomForest rmse 0.105834 trained in 110.06 seconds
5_Default_ExtraTrees rmse 0.106835 trained in 76.58 seconds
There was an error during 6_Default_NearestNeighbors training.
Please check AutoML_1/errors.md for d

AutoML(algorithms=['Random Forest', 'Extra Trees', 'Xgboost', 'CatBoost',
                   'Neural Network', 'Nearest Neighbors'],
       eval_metric='rmse', features_selection=True, golden_features=False,
       hill_climbing_steps=0, mode='Compete', model_time_limit=300,
       stack_models=True, start_random_models=5, top_models_to_improve=3,
       validation_strategy={'k_folds': 3, 'shuffle': True, 'stratify': True,
                            'validation_type': 'kfold'})

In [8]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = automl.predict(test)
test_predictions

array([0.38483122, 0.53658326, 0.52134203, ..., 0.59530087, 0.43779249,
       0.5399273 ])

In [9]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.384831
1,1,0.536583
2,2,0.521342
3,3,0.472984
4,4,0.484761
...,...,...
11995,11995,0.537156
11996,11996,0.478011
11997,11997,0.595301
11998,11998,0.437792


In [10]:
submission.to_csv('lightautoml.csv', index = False)